In [1]:
import os

# Fix: Must be set before any TensorFlow/protobuf-related import happens in the process.
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", "2")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("PYTHONHASHSEED", "42")

import random, gc, warnings

warnings.filterwarnings("ignore")
random.seed(42)
gc.collect()



0

In [2]:
import pandas as pd
import numpy as np
import cv2
from glob import glob
from tqdm import tqdm

TF_AVAILABLE = True
tf = None
K = None
load_model = None

# Fix: import TensorFlow only after env vars are set (cell 1), and keep a clean failure fallback.
try:
    import tensorflow as tf
    from tensorflow.keras import backend as K
    from tensorflow.keras.models import load_model

    print("TF version:", tf.__version__)
except Exception as e:
    TF_AVAILABLE = False
    tf = None
    K = None
    load_model = None
    print(
        "TensorFlow import failed; will write a valid fallback submission. Error:",
        repr(e),
    )



2026-05-13 15:52:22.867447: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778687542.878658      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778687542.882112      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF version: 2.18.0


In [3]:
BATCH_SIZE = 16
EPOCHS = 30
n_splits = 5
fold_selected = 1  # 1..5



In [4]:
BASE_PATH = "../input/uw-madison-gi-tract-image-segmentation"
DEBUG = False

test_df = pd.read_csv(f"{BASE_PATH}/test.csv")
sample_sub = pd.read_csv(f"{BASE_PATH}/sample_submission.csv")

df_train = test_df[["id"]].drop_duplicates().reset_index(drop=True)

df_train["case"] = df_train["id"].apply(
    lambda x: int(x.split("_")[0].replace("case", ""))
)
df_train["day"] = df_train["id"].apply(
    lambda x: int(x.split("_")[1].replace("day", ""))
)
df_train["slice"] = df_train["id"].apply(lambda x: x.split("_")[3])

TEST_DIR = f"{BASE_PATH}/test"
all_test_images = glob(os.path.join(TEST_DIR, "**", "*.png"), recursive=True)
if len(all_test_images) == 0:
    raise RuntimeError(f"No png images found under {TEST_DIR}")

# Fix: avoid fragile string splitting on "/" by using os.path; keep same semantics.
x = os.path.dirname(
    os.path.dirname(os.path.dirname(os.path.dirname(all_test_images[0])))
)  # .../test

df_train["path_partial"] = [
    os.path.join(
        x,
        "case" + str(c),
        "case" + str(c) + "_" + "day" + str(d),
        "scans",
        "slice_" + str(s),
    )
    for c, d, s in zip(
        df_train["case"].values, df_train["day"].values, df_train["slice"].values
    )
]

tmp_df = pd.DataFrame(
    {
        "path_partial": [str(p.rsplit("_", 4)[0]) for p in all_test_images],
        "path": all_test_images,
    }
)

df_train = df_train.merge(tmp_df, on="path_partial", how="left").drop(
    columns=["path_partial"]
)
missing = int(df_train["path"].isna().sum())
if missing:
    raise RuntimeError(
        f"Failed to match {missing} test ids to image paths. Check id parsing/path building."
    )

df_train["width"] = df_train["path"].apply(lambda p: int(p[:-4].rsplit("_", 4)[1]))
df_train["height"] = df_train["path"].apply(lambda p: int(p[:-4].rsplit("_", 4)[2]))

df_train.fillna("", inplace=True)
df_train.head(5)



,id,case,day,slice,path,width,height
0,case123_day20_slice_0001,123,20,0001,../input/uw-madison-gi-tract-image-segmentatio...,266,266
1,case123_day20_slice_0002,123,20,0002,../input/uw-madison-gi-tract-image-segmentatio...,266,266
2,case123_day20_slice_0003,123,20,0003,../input/uw-madison-gi-tract-image-segmentatio...,266,266
3,case123_day20_slice_0004,123,20,0004,../input/uw-madison-gi-tract-image-segmentatio...,266,266
4,case123_day20_slice_0005,123,20,0005,../input/uw-madison-gi-tract-image-segmentatio...,266,266


In [5]:
print(df_train.shape)
if DEBUG:
    df_train = df_train.sample(frac=0.05, random_state=42).reset_index(drop=True)
print(df_train.shape)



(6800, 7)
(6800, 7)


In [6]:
gc.collect()




8

In [7]:
def rle_encode(img):
    """
    img: 2D numpy array, 1 - mask, 0 - background
    Returns run length as string formatted.
    """
    img = img.astype(np.uint8)
    pixels = img.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


def rle_decode(mask_rle, shape, color=1):
    """
    mask_rle: run-length string (start length)
    shape: (height, width, channels)
    Returns numpy array with given shape.
    """
    if (
        mask_rle is None
        or mask_rle == ""
        or (isinstance(mask_rle, float) and np.isnan(mask_rle))
    ):
        return np.zeros(shape, dtype=np.float32)

    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths

    h, w, c = shape
    img = np.zeros((h * w, c), dtype=np.float32)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = color
    return img.reshape((h, w, c), order="F")


def build_masks(labels, input_shape, colors=True):
    height, width = input_shape
    if colors:
        mask = np.zeros((height, width, 3), dtype=np.float32)
        for label in labels:
            mask += rle_decode(label, shape=(height, width, 3), color=np.random.rand(3))
    else:
        mask = np.zeros((height, width, 1), dtype=np.float32)
        for label in labels:
            mask += rle_decode(label, shape=(height, width, 1))
    mask = mask.clip(0, 1)
    return mask




In [8]:
if TF_AVAILABLE:

    class DataGenerator(tf.keras.utils.Sequence):
        def __init__(self, df, batch_size=BATCH_SIZE, subset="train", shuffle=False):
            super().__init__()
            self.df = df.reset_index(drop=True)
            self.shuffle = shuffle
            self.subset = subset
            self.batch_size = batch_size
            self.on_epoch_end()

        def __len__(self):
            return int(np.ceil(len(self.df) / self.batch_size))

        def on_epoch_end(self):
            self.indexes = np.arange(len(self.df))
            if self.shuffle is True:
                np.random.shuffle(self.indexes)

        def __getitem__(self, index):
            batch_indexes = self.indexes[
                index * self.batch_size : (index + 1) * self.batch_size
            ]
            bs = len(batch_indexes)

            X = np.empty((bs, 128, 128, 3), dtype=np.float32)
            y = np.empty((bs, 128, 128, 3), dtype=np.float32)

            for i, idx in enumerate(batch_indexes):
                img_path = self.df["path"].iloc[idx]
                w = int(self.df["width"].iloc[idx])
                h = int(self.df["height"].iloc[idx])

                img = self.__load_grayscale(img_path)  # (128,128,1)
                img3 = np.repeat(img, 3, axis=-1)  # (128,128,3)
                X[i] = img3

                if self.subset == "train":
                    for k, j in zip(
                        [0, 1, 2], ["large_bowel", "small_bowel", "stomach"]
                    ):
                        rles = self.df[j].iloc[idx]
                        masks = rle_decode(rles, shape=(h, w, 1))
                        masks = cv2.resize(
                            masks, (128, 128), interpolation=cv2.INTER_NEAREST
                        )
                        y[i, :, :, k] = masks[:, :, 0]

            if self.subset == "train":
                return X, y
            else:
                return X

        def __load_grayscale(self, img_path):
            img = cv2.imread(img_path, cv2.IMREAD_ANYDEPTH)
            if img is None:
                raise FileNotFoundError(f"Failed to read image: {img_path}")
            img = cv2.resize(img, (128, 128), interpolation=cv2.INTER_AREA)
            img = img.astype(np.float32) / 255.0
            img = np.expand_dims(img, axis=-1)
            return img

else:
    DataGenerator = None



In [9]:
gc.collect()



0

In [10]:
if TF_AVAILABLE:
    from tensorflow.keras.losses import binary_crossentropy

    def dice_coef(y_true, y_pred, smooth=1):
        y_true_f = K.flatten(tf.cast(y_true, tf.float32))
        y_pred_f = K.flatten(tf.cast(y_pred, tf.float32))
        intersection = K.sum(y_true_f * y_pred_f)
        return (2.0 * intersection + smooth) / (
            K.sum(y_true_f) + K.sum(y_pred_f) + smooth
        )

    def iou_coef(y_true, y_pred, smooth=1):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        intersection = K.sum(K.abs(y_true * y_pred), axis=[1, 2, 3])
        union = K.sum(y_true, [1, 2, 3]) + K.sum(y_pred, [1, 2, 3]) - intersection
        iou = K.mean((intersection + smooth) / (union + smooth), axis=0)
        return iou

    def dice_loss(y_true, y_pred):
        smooth = 1.0
        y_true_f = K.flatten(tf.cast(y_true, tf.float32))
        y_pred_f = K.flatten(tf.cast(y_pred, tf.float32))
        intersection = y_true_f * y_pred_f
        score = (2.0 * K.sum(intersection) + smooth) / (
            K.sum(y_true_f) + K.sum(y_pred_f) + smooth
        )
        return 1.0 - score

    def bce_dice_loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        return binary_crossentropy(y_true, y_pred) + 0.5 * dice_loss(y_true, y_pred)

    class FixedDropout(tf.keras.layers.Dropout):
        def _get_noise_shape(self, inputs):
            if self.noise_shape is None:
                return self.noise_shape
            symbolic_shape = K.shape(inputs)
            noise_shape = [
                symbolic_shape[axis] if shape is None else shape
                for axis, shape in enumerate(self.noise_shape)
            ]
            return tuple(noise_shape)

else:
    dice_coef = None
    iou_coef = None
    bce_dice_loss = None
    FixedDropout = None



In [11]:
gc.collect()



0

In [12]:
model = None
if TF_AVAILABLE:
    custom_objects = {
        "FixedDropout": FixedDropout,
        "dice_coef": dice_coef,
        "iou_coef": iou_coef,
        "bce_dice_loss": bce_dice_loss,
    }

    MODEL_CANDIDATES = [
        "../input/uwmgi-unet-keras/model.h5",
        "../input/uwmgi-unet-keras/model.keras",
        "../input/uwmgi-unet-keras/best_model.h5",
        "../input/uwmgi-unet-keras/best_model.keras",
    ]
    for mp in MODEL_CANDIDATES:
        if os.path.exists(mp):
            try:
                model = load_model(mp, custom_objects=custom_objects, compile=False)
                print("Loaded model from:", mp)
                break
            except Exception as e:
                print("Model load failed for", mp, "error:", repr(e))
                model = None

    if model is None:
        extra = []
        extra += glob("../input/**/model.h5", recursive=True)
        extra += glob("../input/**/model.keras", recursive=True)
        extra += glob("../input/**/best_model.h5", recursive=True)
        extra += glob("../input/**/best_model.keras", recursive=True)
        seen = set()
        for mp in extra:
            if mp in seen:
                continue
            seen.add(mp)
            try:
                model = load_model(mp, custom_objects=custom_objects, compile=False)
                print("Loaded model from:", mp)
                break
            except Exception:
                model = None

print("Loaded model:", model is not None)
gc.collect()



Loaded model: False


0

In [13]:
if TF_AVAILABLE and model is not None:
    pred_batches = DataGenerator(df_train, batch_size=1, subset="test", shuffle=False)
    gc.collect()
    LOGITS = model.predict(pred_batches, verbose=1)
else:
    LOGITS = np.zeros((len(df_train), 128, 128, 3), dtype=np.float32)

if LOGITS.shape[0] < len(df_train):
    pad = np.zeros((len(df_train) - LOGITS.shape[0], 128, 128, 3), dtype=LOGITS.dtype)
    LOGITS = np.concatenate([LOGITS, pad], axis=0)
elif LOGITS.shape[0] > len(df_train):
    LOGITS = LOGITS[: len(df_train)]

LOGITS = np.clip(LOGITS, 0.0, 1.0)
gc.collect()



0

In [14]:
len(LOGITS)



6800

In [15]:
lbs, sbs, sts = [], [], []
for index, row in tqdm(df_train.iterrows(), total=df_train.shape[0]):
    h = int(df_train.iloc[index]["height"])
    w = int(df_train.iloc[index]["width"])

    pred0 = cv2.resize(LOGITS[index, :, :, 0], (w, h), interpolation=cv2.INTER_NEAREST)
    pred1 = cv2.resize(LOGITS[index, :, :, 1], (w, h), interpolation=cv2.INTER_NEAREST)
    pred2 = cv2.resize(LOGITS[index, :, :, 2], (w, h), interpolation=cv2.INTER_NEAREST)

    pred0 = (pred0 >= 0.5).astype("uint8")
    pred1 = (pred1 >= 0.5).astype("uint8")
    pred2 = (pred2 >= 0.5).astype("uint8")

    lbs.append(rle_encode(pred0))
    sbs.append(rle_encode(pred1))
    sts.append(rle_encode(pred2))

del LOGITS
gc.collect()



100%|██████████| 6800/6800 [00:04<00:00, 1491.18it/s]


0

In [16]:
pred_map = {}
for i, rid in enumerate(df_train["id"].values):
    pred_map[(rid, "large_bowel")] = lbs[i]
    pred_map[(rid, "small_bowel")] = sbs[i]
    pred_map[(rid, "stomach")] = sts[i]

sub = sample_sub.copy()
sub["predicted"] = [
    pred_map.get((rid, cls), "")
    for rid, cls in zip(sub["id"].values, sub["class"].values)
]

sub["predicted"] = sub["predicted"].fillna("")
sub.to_csv("submission.csv", index=False)

assert sub.shape[0] == sample_sub.shape[0]
assert list(sub.columns) == ["id", "class", "predicted"]
print(sub.head())



                         id        class predicted
0  case123_day20_slice_0001  large_bowel          
1  case123_day20_slice_0001  small_bowel          
2  case123_day20_slice_0001      stomach          
3  case123_day20_slice_0002  large_bowel          
4  case123_day20_slice_0002  small_bowel          


In [17]:
sub.head()

,id,class,predicted
0,case123_day20_slice_0001,large_bowel,
1,case123_day20_slice_0001,small_bowel,
2,case123_day20_slice_0001,stomach,
3,case123_day20_slice_0002,large_bowel,
4,case123_day20_slice_0002,small_bowel,
